# Testing Fine-Tuned Resume-Job Matching Model on Kaggle Recruitment Dataset

## Overview
This notebook tests our fine-tuned BERT model on the **Kaggle Recruitment Dataset**.

### Dataset Information
- **Source**: Kaggle - surendra365/recruitement-dataset
- **Contains**: Applicant details, resumes, job descriptions, and matching labels
- **Best Match Column**: Score/label indicating how well applicant matches job role

### Model Information
- **Model**: sentence-transformers/all-mpnet-base-v2 (768 dim)
- **Optimized for**: Mac (Apple Silicon)
- **Output**: Similarity score (0-1) indicating job fit

## 1. Install Required Packages

In [1]:
!pip install transformers torch pandas numpy scikit-learn kagglehub accelerate scipy

zsh:1: command not found: pip


## 2. Import Libraries

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS (Mac GPU) available: {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PyTorch version: 2.8.0
CUDA available: False
MPS (Mac GPU) available: True
Using device: mps


## 3. Download Kaggle Dataset

In [ ]:
print("Downloading Kaggle recruitment dataset...")
path = kagglehub.dataset_download("surendra365/recruitement-dataset")
print(f"Path to dataset files: {path}")

Path to dataset files: /Users/mac/.cache/kagglehub/datasets/surendra365/recruitement-dataset/versions/2


## 4. Load and Explore Dataset

In [14]:
# Find CSV files
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f"CSV files found: {csv_files}")

# Load dataset
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

print(f"\nBasic statistics:")
df.tail()

CSV files found: ['job_applicant_dataset.csv']

Dataset shape: (10000, 9)

Columns: ['Job Applicant Name', 'Age', 'Gender', 'Race', 'Ethnicity', 'Resume', 'Job Roles', 'Job Description', 'Best Match']

First few rows:

Basic statistics:


,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
9995,Jada Williams,30,Female,Negroid/Black,Ghanaian,"Proficient in Biology, Regulatory Compliance, ...",Biomedical Engineer,A Biomedical Engineer designs and develops med...,0
9996,Jaden Carter,52,Male,Negroid/Black,Nigerian,"Proficient in Communication, Teamwork, Lesson ...",Teacher,A Teacher shapes the future of students by del...,0
9997,Mia Foster,25,Female,White/Caucasian,German,"Proficient in Medical Terminology, Critical Th...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
9998,Stella Green,51,Female,White/Caucasian,Irish,"Proficient in Exercise Programming, Motivation...",Fitness Coach,A Fitness Coach is responsible for helping cl...,1
9999,Ryo Nishida,46,Male,Mongoloid/Asian,Thai,"Proficient in Content Strategy, Copywriting, C...",Content Writer,"As a Content Writer, you will create written m...",0


## 5. Data Preprocessing

In [6]:
# Clean data
df_clean = df.dropna(subset=['Resume', 'Job Description', 'Best Match']).copy()
df_clean['Resume'] = df_clean['Resume'].astype(str)
df_clean['Job Description'] = df_clean['Job Description'].astype(str)

# Process Best Match column
def process_best_match(match_value):
    match_str = str(match_value).strip()
    try:
        score = float(match_str)
        if 0 <= score <= 1:
            return score
        elif 0 <= score <= 100:
            return score / 100.0
        elif 0 <= score <= 10:
            return score / 10.0
        else:
            return np.clip(score, 0.0, 1.0)
    except ValueError:
        match_lower = match_str.lower()
        if any(w in match_lower for w in ['excellent', 'perfect', 'strong', 'high', 'good', 'best']):
            return 1.0
        elif any(w in match_lower for w in ['moderate', 'medium', 'fair', 'potential', 'average']):
            return 0.5
        elif any(w in match_lower for w in ['poor', 'low', 'weak', 'no', 'bad']):
            return 0.0
        else:
            return 0.5

df_clean['similarity_score'] = df_clean['Best Match'].apply(process_best_match)
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"\nScore distribution:\n{df_clean['similarity_score'].value_counts().sort_index()}")

Cleaned dataset shape: (10000, 10)

Score distribution:
similarity_score
0.0    5150
1.0    4850
Name: count, dtype: int64


## 6. Prepare Data for Training

In [7]:
# Split data
train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 8000, Val: 1000, Test: 1000


## 7. Load and Train Model

In [8]:
# Load model
model_name = "sentence-transformers/all-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, problem_type="regression")
model.to(device)
print(f"Model loaded on {device}")

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on mps


In [9]:
# Tokenize
def tokenize_fn(examples):
    tok = tokenizer(examples['Job Description'], examples['Resume'], padding='max_length', truncation=True, max_length=512)
    tok['labels'] = examples['similarity_score']
    return tok

train_ds = Dataset.from_pandas(train_df[['Job Description', 'Resume', 'similarity_score']])
val_ds = Dataset.from_pandas(val_df[['Job Description', 'Resume', 'similarity_score']])
test_ds = Dataset.from_pandas(test_df[['Job Description', 'Resume', 'similarity_score']])

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=['Job Description', 'Resume', 'similarity_score'])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=['Job Description', 'Resume', 'similarity_score'])
test_tok = test_ds.map(tokenize_fn, batched=True, remove_columns=['Job Description', 'Resume', 'similarity_score'])
print("Tokenization complete")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenization complete


In [10]:
# Metrics
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.clip(preds.squeeze(), 0.0, 1.0)
    mse = mean_squared_error(labels, preds)
    pearson, _ = pearsonr(labels, preds)
    spearman, _ = spearmanr(labels, preds)
    return {'mse': mse, 'pearson': pearson, 'spearman': spearman}

In [11]:
# Train
use_mps = device.type == "mps"
args = TrainingArguments(
    output_dir="./kaggle_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mse",
    report_to="none",
    use_mps_device=use_mps
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok, compute_metrics=compute_metrics)
print("Training...")
trainer.train()
trainer.save_model("./kaggle_model")
tokenizer.save_pretrained("./kaggle_model")
print("Training complete!")

Training...


Epoch,Training Loss,Validation Loss,Mse,Pearson,Spearman
1,0.249000,0.240510,0.240328,0.207816,0.108694
2,0.243500,0.240995,0.240818,0.203193,0.112963
3,0.244100,0.238801,0.238801,0.210725,0.135708


Training complete!


## 8. Evaluate

In [12]:
results = trainer.evaluate(test_tok)
print(f"\nTest Results:")
print(f"MSE: {results['eval_mse']:.4f}")
print(f"Pearson: {results['eval_pearson']:.4f}")
print(f"Spearman: {results['eval_spearman']:.4f}")


Test Results:
MSE: 0.2490
Pearson: 0.1178
Spearman: 0.0372
